# Tomahawk Rev2.4 integrated screening

This notebook runs the current Rev2.4 provisional inputs through a fixed-boundary axisymmetric Grad-Shafranov screen, D-T power/confinement balance, coolant conservation screen, and reduced uncertainty envelope. Attach the matching Rev2.4 Python runner and provisional_inputs data ZIP as Kaggle inputs; the bootstrap also accepts a combined package ZIP. It runs a smoke case first and then the full case count. Results remain preliminary and do not claim 3-D MHD, OpenMC, CFD/CHT, CHF, ignition, or hardware validation.

In [ ]:
import importlib.util
import os
from pathlib import Path
import zipfile

KAGGLE_INPUT_ROOT = Path(os.environ.get("TOMAHAWK_KAGGLE_INPUT_ROOT", "/kaggle/input")).resolve()
KAGGLE_WORKING_ROOT = Path(os.environ.get("TOMAHAWK_KAGGLE_WORKING_ROOT", "/kaggle/working")).resolve()
OUT = Path(os.environ.get("TOMAHAWK_OUTPUT_ROOT", str(KAGGLE_WORKING_ROOT / "tomahawk_rev24_screen"))).resolve()
KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

def unique_existing(paths):
    result = []
    for path in paths:
        path = Path(path)
        if path.exists() and path not in result:
            result.append(path)
    return result

roots = unique_existing((Path.cwd(), KAGGLE_INPUT_ROOT, KAGGLE_WORKING_ROOT))

def safe_extract(archive, target):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        for member in handle.infolist():
            destination = (target / member.filename).resolve()
            if destination != target and target not in destination.parents:
                raise ValueError(f"Unsafe archive member: {member.filename}")
        handle.extractall(target)

archive_names = {
    "Tomahawk_Rev2_4_Integrated_Screening_Package.zip",
    "tomahawk_rev24_confinement_handoff.zip",
    "tomahawk_rev24_data_inputs.zip",
}
archives = []
for root in roots:
    if not root.is_dir():
        continue
    try:
        for archive in root.rglob("*.zip"):
            name = archive.name.lower()
            if archive.name in archive_names or (
                "tomahawk" in name and any(token in name for token in ("package", "handoff", "data_inputs"))
            ):
                archives.append(archive)
    except OSError:
        pass

for archive in sorted(set(archives), key=lambda path: str(path)):
    target = KAGGLE_WORKING_ROOT / "tomahawk_rev24_inputs" / archive.stem
    safe_extract(archive, target)
    roots.append(target)

script_candidates = []
for root in roots:
    if root.is_dir():
        try:
            script_candidates.extend(root.rglob("Tomahawk_Rev2_4_Integrated_Screening.py"))
        except OSError:
            pass
if not script_candidates:
    raise FileNotFoundError(
        "No Rev2.4 Python runner found. Attach the .py file or a Rev2.4 package ZIP.")
SCRIPT = sorted(set(script_candidates), key=lambda path: (
    "/kaggle/input" not in str(path), len(str(path)), str(path)))[0].resolve()

def is_input_root(path):
    path = Path(path)
    return (path / "MHD" / "mhd_profile_sweep.csv").is_file()

input_candidates = []
for root in roots:
    if not root.is_dir():
        continue
    if is_input_root(root):
        input_candidates.append(root)
    try:
        for profile in root.rglob("mhd_profile_sweep.csv"):
            if profile.parent.name == "MHD" and is_input_root(profile.parent.parent):
                input_candidates.append(profile.parent.parent)
    except OSError:
        pass

explicit_input_root = os.environ.get("TOMAHAWK_INPUT_ROOT")
if explicit_input_root:
    INPUT_ROOT = Path(explicit_input_root).expanduser().resolve()
elif is_input_root(SCRIPT.parent / "provisional_inputs"):
    INPUT_ROOT = (SCRIPT.parent / "provisional_inputs").resolve()
elif input_candidates:
    INPUT_ROOT = sorted(set(input_candidates), key=lambda path: (
        "/kaggle/input" not in str(path), len(str(path)), str(path)))[0].resolve()
else:
    raise FileNotFoundError(
        "No provisional_inputs directory found. Attach tomahawk_rev24_data_inputs.zip "
        "or a package containing provisional_inputs/.")

spec = importlib.util.spec_from_file_location("tomahawk_rev24_screen", SCRIPT)
screen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen)
print("Runner:", getattr(screen, "VERSION", "unknown"))
print("Script:", SCRIPT)
print("Inputs:", INPUT_ROOT)
print("Output:", OUT)

## Smoke run

The smoke run checks imports, profile closure, the sparse equilibrium solve, the coolant-row parser, strict JSON writing, and the explicit blocker policy.

In [ ]:
smoke = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "smoke",
    label="smoke",
    cases=int(os.environ.get("TOMAHAWK_SMOKE_CASES", "256")),
    seed=20260905,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
smoke_checks = {
    "overall_status": smoke["overall_status"],
    "GS_residual_pass": smoke["gates"]["GS_residual_pass"],
    "profile_consistency_pass": smoke["gates"]["profile_consistency_pass"],
    "reference_self_heating_reconciliation_pass": smoke["gates"]["reference_self_heating_reconciliation_pass"],
    "alpha_self_heating_pass": smoke["gates"]["alpha_self_heating_pass"],
    "tau_E_margin_pass": smoke["gates"]["tau_E_margin_pass"],
    "confinement_feasibility_screen_pass": smoke["gates"]["confinement_feasibility_screen_pass"],
    "coolant_screen_pass": smoke["gates"]["coolant_screen_pass"],
}
print(smoke_checks)
if not (
    smoke_checks["overall_status"] == "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"
    and smoke_checks["GS_residual_pass"]
    and smoke_checks["profile_consistency_pass"]
    and smoke_checks["reference_self_heating_reconciliation_pass"]
    and smoke_checks["alpha_self_heating_pass"]
    and smoke_checks["tau_E_margin_pass"]
    and smoke_checks["confinement_feasibility_screen_pass"]
    and smoke_checks["coolant_screen_pass"]
):
    raise RuntimeError(f"Smoke gate failed: {smoke_checks}")
print("SMOKE_PASS: numerical and coolant screens are green; external solver blockers remain explicit.")

## Full run

The full run repeats the auditable solve and evaluates the reduced 65,536-case parameter envelope. The uncertainty cases are scaled screens and do not re-solve Grad-Shafranov for every sample.

In [ ]:
full = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "full",
    label="full",
    cases=int(os.environ.get("TOMAHAWK_FULL_CASES", "65536")),
    seed=20260906,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
full_checks = {
    "overall_status": full["overall_status"],
    "confinement_numerical_screen_pass": full["gates"]["confinement_numerical_screen_pass"],
    "fusion_GW_module": full["confinement_metrics"]["fusion_GW_module"],
    "alpha_self_heating_ratio": full["confinement_metrics"]["alpha_self_heating_ratio"],
    "reference_condition_self_heating_ratio": full["confinement_metrics"]["reference_condition_self_heating_ratio"],
    "reference_self_heating_reconciliation_pass": full["gates"]["reference_self_heating_reconciliation_pass"],
    "coolant_screen_pass": full["gates"]["coolant_screen_pass"],
    "all_gate_pass_percent": full["uncertainty_summary"]["all_gate_pass_percent"],
}
print(full_checks)
if not (
    full_checks["overall_status"] == "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"
    and full_checks["confinement_numerical_screen_pass"]
    and full_checks["reference_self_heating_reconciliation_pass"]
    and full_checks["coolant_screen_pass"]
):
    raise RuntimeError(f"Full-run gate failed: {full_checks}")
print("FULL_SCREEN_PASS: external solver blockers remain explicit.")

In [ ]:
import json
for name in ("equilibrium_metrics.json", "confinement_metrics.json", "coolant_summary.json", "gate_results.json"):
    path = OUT / "full" / name
    print("\n", name)
    print(json.dumps(json.loads(path.read_text()), indent=2)[:6000])

bundle = screen.zip_directory(OUT)
print("\nBundle:", bundle)
print("Plots:", sorted(str(path) for path in (OUT / "full").glob("*.png")))

## Interpretation

A fixed-boundary numerical pass confirms only that the declared finite-difference equation was solved to the declared residual. The active tau-E screen is 2.70 s; the recorded 1.36 s / 63% point is retained as a separately reconciled reference condition. The full-incident heat row routes 50% to secondary systems and keeps 50% cassette-local; secondary-system capacity remains unqualified. The numerical and coolant screens can pass while free-boundary 3-D MHD, orbit-resolved alpha transport, nuclear transport, CFD/CHT, CHF, and hardware validation remain external blockers. Treat the generated bundle as a reproducible screening artifact, not as an engineering validation package.